# EAAI Phase 2: paired TechQA reranking and adaptive invocation

This notebook executes the committed prospective protocol. It never edits prior scientific artifacts. Qwen is the primary held-out experiment; Mistral is an optional secondary replication. Select a GPU runtime before starting.

The private baseline bundle contains controlled benchmark-derived artifacts. Keep it in private Google Drive storage and do not publish it.

In [ ]:
import os
import subprocess
from pathlib import Path

gpu = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
    capture_output=True, text=True, check=False,
)
if gpu.returncode != 0:
    raise RuntimeError('No GPU detected. Choose Runtime > Change runtime type > GPU.')
print(gpu.stdout.strip())

from google.colab import drive
drive.mount('/content/drive')

REPOSITORY_URL = 'https://github.com/kuromi1kow/chunkrag-course-project.git'
IMPLEMENTATION_COMMIT = 'ed4aabf2427015773442f01b084647697ce2a222'
REPO = Path('/content/chunkrag-eaai-phase2')
DRIVE_ROOT = Path('/content/drive/MyDrive/chunkrag_outputs/eaai_phase2')
BASELINE_ARCHIVE = DRIVE_ROOT / 'eaai_phase2_baseline_private.tar.gz'
BASELINE_SHA256 = DRIVE_ROOT / 'eaai_phase2_baseline_private.tar.gz.sha256'
RUN_MISTRAL = False
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)


In [ ]:
import shutil

if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(['git', 'clone', REPOSITORY_URL, str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', IMPLEMENTATION_COMMIT], check=True)
actual = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
assert actual == IMPLEMENTATION_COMMIT, (actual, IMPLEMENTATION_COMMIT)
print(f'Pinned implementation: {actual}')


In [ ]:
import sys

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO / 'requirements-eaai-phase2.txt')],
    check=True,
)
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--ignore-requires-python', '-e', str(REPO)],
    check=True,
)
os.environ['PYTHONPATH'] = str(REPO / 'src')
os.environ['HF_HOME'] = str(DRIVE_ROOT / 'huggingface_cache')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
print('Dependencies installed; Hugging Face cache persists in Drive.')


## Restore and verify the frozen baseline

Create the private archive locally with `PYTHONPATH=src python3 scripts/prepare_eaai_phase2_colab_bundle.py`, then place the `.tar.gz` and `.sha256` files at the Drive paths configured above.

In [ ]:
import hashlib

if not BASELINE_ARCHIVE.is_file() or not BASELINE_SHA256.is_file():
    raise FileNotFoundError(
        f'Upload the private baseline archive and sidecar to {DRIVE_ROOT} before continuing.'
    )
expected_hash = BASELINE_SHA256.read_text().split()[0]
digest = hashlib.sha256()
with BASELINE_ARCHIVE.open('rb') as handle:
    for block in iter(lambda: handle.read(1024 * 1024), b''):
        digest.update(block)
actual_hash = digest.hexdigest()
assert actual_hash == expected_hash, (actual_hash, expected_hash)
subprocess.run(['tar', '-xzf', str(BASELINE_ARCHIVE), '-C', str(REPO)], check=True)
subprocess.run(
    [sys.executable, str(REPO / 'scripts/verify_eaai_phase2_baseline.py')],
    cwd=REPO, check=True, env=os.environ.copy(),
)


In [ ]:
# Keep resumable private rows in Drive while preserving required repository paths.
run_id = 'techqa_adaptive_v1'
for kind in ('results', 'artifacts'):
    local = REPO / kind / 'eaai_phase2' / run_id
    remote = DRIVE_ROOT / kind / 'eaai_phase2' / run_id
    remote.mkdir(parents=True, exist_ok=True)
    local.parent.mkdir(parents=True, exist_ok=True)
    if local.is_symlink():
        local.unlink()
    elif local.exists():
        raise RuntimeError(f'Refusing to replace existing non-symlink output: {local}')
    local.symlink_to(remote, target_is_directory=True)
print(f'Resumable outputs: {DRIVE_ROOT}')


In [ ]:
runner = [sys.executable, str(REPO / 'scripts/run_eaai_phase2.py'), '--device', 'cuda']
subprocess.run([*runner, 'dry-run'], cwd=REPO, check=True, env=os.environ.copy())
subprocess.run([*runner, 'partition'], cwd=REPO, check=True, env=os.environ.copy())


## Development stage

The next two cells create paired development retrieval and Qwen generations. They checkpoint each question--chunker row in Drive and are safe to rerun after a disconnect.

In [ ]:
subprocess.run(
    [*runner, 'retrieve', '--split', 'development'],
    cwd=REPO, check=True, env=os.environ.copy(),
)


In [ ]:
subprocess.run(
    [*runner, 'generate', '--split', 'development', '--generator', 'qwen'],
    cwd=REPO, check=True, env=os.environ.copy(),
)


In [ ]:
# This freezes the fixed-threshold gate before any held-out artifact can be created.
subprocess.run([*runner, 'fit-gate'], cwd=REPO, check=True, env=os.environ.copy())


## Held-out Qwen stage

Do not inspect or alter the gate after this point. The primary analysis uses 200 questions as the sampling unit and runs one prespecified sign-flip test.

In [ ]:
subprocess.run(
    [*runner, 'retrieve', '--split', 'heldout_test'],
    cwd=REPO, check=True, env=os.environ.copy(),
)


In [ ]:
subprocess.run(
    [*runner, 'generate', '--split', 'heldout_test', '--generator', 'qwen'],
    cwd=REPO, check=True, env=os.environ.copy(),
)


In [ ]:
subprocess.run(
    [sys.executable, str(REPO / 'scripts/analyze_eaai_phase2.py')],
    cwd=REPO, check=True, env=os.environ.copy(),
)
primary = DRIVE_ROOT / 'results' / 'eaai_phase2' / run_id / 'primary_confirmatory_analysis.json'
print(primary)
print(primary.read_text()[:4000])


## Optional Mistral replication

Use an L4 or A100 runtime when possible. This stage applies the already frozen Qwen-trained gate without refitting and does not add a confirmatory p-value.

In [ ]:
if RUN_MISTRAL:
    subprocess.run([*runner, 'run-mistral'], cwd=REPO, check=True, env=os.environ.copy())
    subprocess.run(
        [sys.executable, str(REPO / 'scripts/analyze_eaai_phase2.py'), '--include-mistral'],
        cwd=REPO, check=True, env=os.environ.copy(),
    )
else:
    print('Mistral replication skipped; set RUN_MISTRAL=True only on a suitable GPU runtime.')
